# A demo of the pyINSPECTA package

In [ ]:
from __future__ import annotations

%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["figure.dpi"] = 150

from pyINSPECTA import SDHDF
from pyINSPECTA.logger import logger

logger.setLevel("INFO")

#### Let's load an SDHDF file - it is represented by an SDHDF class instance (set verbose=True to get all the metadata):

In [ ]:
fname = "/Users/tho822/Documents/P1123/uwl_240723_084845_0.flag.atflagged.Fmean.hdf"

In [ ]:
sdhdf = SDHDF(fname, parallel=True)

#### The SDHDF class contains data on the beams and metadata:

In [ ]:
print(sdhdf.beams)

In [ ]:
print(sdhdf.metadata)

#### Let's explore the observation metadata....

In [ ]:
sdhdf.print_obs_metadata()

#### Now the observation configuration....

In [ ]:
sdhdf.print_obs_config()

#### We can access the observation parameters with...

In [ ]:
sdhdf.metadata.obs_params

#### ...where the HDF attributes can be accessed with: 

In [ ]:
sdhdf.metadata.obs_params

#### The beam data can be accessed from the beam name (or beams list):

In [ ]:
print(sdhdf.beam_0)
# or
# print(f.beams[0])

#### The Beam class contains the subband data - these can be accessed in a similar way to the beams:

In [ ]:
sdhdf.beam_0.subbands

In [ ]:
# print(sdhdf.beam_0.subbands[0].)
# or
print(sdhdf.beam_0.subbands[0])

#### At the bottom level, the data itself is held as either an `xarray.Dataset` or `pandas.DataFrame` depending on the data type:

In [ ]:
sdhdf.beam_0.subbands[0].astronomy_dataset

In [ ]:
sdhdf.beam_0.subbands[0].astronomy_dataset.data

#### For example, using the `xarray.DataArray` we have a lot of power to visualise and manipulate the data - see the [xarray documentation](https://docs.xarray.dev/en/stable/user-guide/data-structures.html) for more information.

#### I've also implemented some commonly-used commands on methods inside the dataclasses. For example, we can make a waterfall plot, plot a spectrum, and inspect the metadata. This is all done with `xarray` or `pandas` under the hood - so much more complex investigations are possible using those tools.

#### Here's a waterfall plot, which can be called from the `SDHDF` and `Beam` classes, but is really calling the base method on the `Subband` class:

In [ ]:
sdhdf.plot_waterfall(
    beam=0,
    subband=0,
    polarization=0,
    flag=False,
    norm=plt.cm.colors.LogNorm(vmin=1e0),
    y="ELAPSED_TIME",
)

Note that this is just a wrapper around the methods in `xarray`. You can use these directly for powerful visualisation and processing functionality

In [ ]:
sdhdf.beam_0.subbands[0].astronomy_dataset.isel(polarization=0).data.plot(
    norm=plt.cm.colors.LogNorm(vmin=1e0, vmax=1e4), x="frequency", y="ELAPSED_TIME"
)

#### And a similar plot for the spectrum:

In [ ]:
ax = sdhdf.plot_spectrum(beam=0, subband=0, x="frequency", flag=True)
ax.set_yscale("log")

#### A wide-band plot can be called from the `SDHDF` object, but it also really just calling down to the `Beam` class:

In [ ]:
ax = sdhdf.plot_wide(beam=0, polarization=0, x="frequency", flag=True)
ax.set_yscale("log")

#### Writing the data out back to disk isn't possible, yet...

In [ ]:
sdhdf.write("test.hdf5")

#### RFI flagging routines have been implemented though:

In [ ]:
sdhdf.auto_flag_rfi(sigma=3, n_windows=100)

In [ ]:
ax = sdhdf.plot_wide(beam=0, polarization=0, flag=True, x="frequency")
ax.set_yscale("log")

In [ ]:
ax = sdhdf.plot_waterfall(
    beam=0,
    subband=0,
    polarization=0,
    flag=True,
    norm=plt.cm.colors.LogNorm(vmin=1e0),
    y="ELAPSED_TIME",
)